# CMU-MOSEI · 1. Download the selected CSD streams

Use a **CPU runtime** for this notebook to conserve your Colab compute units.
It downloads the four required streams from `samarwarsi/cmu-mosei`, version 1
(about 28.4 GB), into your own Google Drive. Alternative FACET/GloVe/phone
streams are not inputs to this model. The CSD package contains features,
words and labels, rather than raw videos.

First build `dist/mosei-colab-source.zip` in the local project with
`.venv/bin/python scripts/build_colab_bundle.py`. Upload that small archive
below. No dataset or model weights are inside it.

Drive and Colab authentication are performed interactively by you. Nothing
is shared publicly. Downloads resume from saved chunks if a session ends.


In [ ]:
from pathlib import Path

from google.colab import drive, files

drive.mount("/content/drive")
STORE = Path("/content/drive/MyDrive/NLP_Assignment_3")
STORE.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()  # Choose mosei-colab-source.zip only.
if len(uploaded) != 1:
    raise ValueError("Upload exactly one source bundle")
name, content = next(iter(uploaded.items()))
BUNDLE = STORE / "mosei-colab-source.zip"
BUNDLE.write_bytes(content)
del uploaded, content

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

PROJECT = Path("/content/mosei-project")
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    for item in archive.infolist():
        target = (PROJECT / item.filename).resolve()
        if not target.is_relative_to(PROJECT.resolve()):
            raise ValueError("Unsafe source archive path")
    archive.extractall(PROJECT)
hashes = json.loads((PROJECT / "bundle-manifest.json").read_text())
for name, expected in hashes.items():
    if hashlib.sha256((PROJECT / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f"Source bundle checksum mismatch: {name}")
os.chdir(PROJECT)
os.environ.update(USE_TF="0", USE_FLAX="0", TOKENIZERS_PARALLELISM="false")
tools = json.loads((PROJECT / "data/tools/video-tools.json").read_text())
versions = [f"{name}=={version}" for name, version in tools["manifest"]["software"].items()]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[video]", *versions], check=True
)
print("Installed the source bundle and matching feature-library versions.")

In [ ]:
source = STORE / "csd"
subprocess.run([sys.executable, "scripts/download_csd.py", "--output", str(source)], check=True)
shutil.copy2(PROJECT / "data/csd/splits.json", source / "splits.json")
print("Download complete. Open 02_train_colab.ipynb in an A100 GPU runtime when available.")

Once the download completes, disconnect this CPU runtime. The files remain
in `MyDrive/NLP_Assignment_3/csd`. If interrupted, rerun the cells; complete
files are checked and partial downloads resume. Your source bundle also
remains in that Drive folder for the training notebook.
